In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torchvision.models as models
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
import numpy as np
import os

# ---------------- Setup ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == 'cuda':
    cudnn.benchmark = True

# ---------------- Hyperparameters ----------------
BATCH_SIZE   = 32
EPOCHS       = 50
INITIAL_LR   = 0.001
WEIGHT_DECAY = 1e-4
NUM_CLASSES  = 7
RANDOM_SEED  = 42

In [2]:
DATA_DIR = "/kaggle/input/fer2013"
# ---------------- Transforms ----------------
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ---------------- Datasets ----------------
train_val_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, "train"), transform=train_transforms)
test_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, "test"), transform=val_test_transforms)

train_size = int(0.75 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(
    train_val_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

# ---------------- Data Loaders ----------------
num_workers = min(4, os.cpu_count() if os.cpu_count() is not None else 1)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=num_workers, pin_memory=(DEVICE.type == 'cuda'))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=num_workers, pin_memory=(DEVICE.type == 'cuda'))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=num_workers, pin_memory=(DEVICE.type == 'cuda'))

# ---------------- Class Weights ----------------
all_labels = [sample[1] for sample in train_val_dataset.samples]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_labels),
    y=all_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torchvision import models
from tqdm import tqdm

def build_model(num_classes, device):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )
    return model.to(device)

def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device, epoch, total_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    loader = tqdm(dataloader, desc=f"Epoch {epoch}/{total_epochs}", unit='batch')
    
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)
        loader.set_postfix(loss=running_loss/total, acc=correct/total)

    return running_loss / total, correct / total

def validate(model, dataloader, criterion, device):
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in dataloader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
            val_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == lbls).sum().item()
            val_total += lbls.size(0)

    return val_loss / val_total, val_correct / val_total

def train_model(train_loader, val_loader, num_classes, class_weights, device,
                initial_lr=1e-3, weight_decay=1e-4, epochs=10, save_path="cnn_best.pth"):
    
    model = build_model(num_classes, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.fc.parameters(), lr=initial_lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()
    best_val_loss = float('inf')

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, epoch, epochs
        )
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch}/{epochs} | LR: {current_lr:.6f} | "
              f"Train: loss={train_loss:.4f}, acc={train_acc:.4f} | "
              f"Val: loss={val_loss:.4f}, acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print("✅ Saved best model")

    return model


In [4]:
model = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=NUM_CLASSES,
    class_weights=class_weights,
    device=DEVICE,
    initial_lr=INITIAL_LR,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    save_path="cnn_nour.pth"
)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 202MB/s]
/tmp/ipykernel_19/3915128970.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch

Epoch 1/50 | LR: 0.000999 | Train: loss=1.8578, acc=0.2433 | Val: loss=1.7479, acc=0.3451
✅ Saved best model


Epoch 2/50: 100%|██████████| 673/673 [00:44<00:00, 15.04batch/s, acc=0.303, loss=1.78]


Epoch 2/50 | LR: 0.000996 | Train: loss=1.7810, acc=0.3026 | Val: loss=1.7429, acc=0.3770
✅ Saved best model


Epoch 3/50: 100%|██████████| 673/673 [00:43<00:00, 15.64batch/s, acc=0.309, loss=1.77]


...

Epoch 48/50 | LR: 0.000004 | Train: loss=1.6212, acc=0.3628 | Val: loss=1.5662, acc=0.4043


Epoch 49/50: 100%|██████████| 673/673 [00:45<00:00, 14.92batch/s, acc=0.365, loss=1.62]


Epoch 49/50 | LR: 0.000001 | Train: loss=1.6162, acc=0.3645 | Val: loss=1.5647, acc=0.4156
✅ Saved best model


Epoch 50/50: 100%|██████████| 673/673 [00:46<00:00, 14.45batch/s, acc=0.365, loss=1.61]


Epoch 50/50 | LR: 0.000000 | Train: loss=1.6120, acc=0.3645 | Val: loss=1.5715, acc=0.4037


In [5]:
model.load_state_dict(torch.load("cnn_nour.pth"))
model.eval()
correct = total = 0
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        with autocast():
            outputs = model(imgs)
        preds = outputs.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)

test_acc = correct / total
print(f"\n🎯 Final Test Accuracy: {test_acc * 100:.2f}%")

/tmp/ipykernel_19/2587316473.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("cnn_nour.pth"))
/tmp/ipykernel_19/2587316473.py:7: FutureW


🎯 Final Test Accuracy: 40.85%
